In [1]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
sys.path.insert(0, str(EXT / "diffmot"))               # agar from external.* resolve
sys.path.insert(0, str(EXT / "diffmot" / "external"))  # agar import fast_reid / YOLOX resolve (fast_reid tidak punya setup.py)
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

ROOT : /opt/shared_data/hibah-riset
DATA : /opt/shared_data/hibah-riset/data/s2
python: /home/if2011/miniconda3/envs/jupyterhub-env/bin/python


# 40 — DiffMOT: Patch Eval + Smoke ReID

**Kernel: `s2-diffmot`** (python 3.9 + torch 2.0.1 cu118). Pastikan kernel sudah dipilih.

Alasan patch: di `diffmot.py` eval, `cv2.imread` dikomentari → `compute_embedding(img=None,...)`
crash bila cache `{reid_dir}/{seq}_embedding.pkl` belum ada. Patch 2 baris: baca img dan
teruskan ke `tracker.update(...)` — cache terisi otomatis saat run pertama, reuse setelahnya.
`scripts/s2/patch_diffmot_eval.py` idempotent.

Smoke test: load bobot ReID (FastReID) + forward dummy — validasi CUDA/weight SEBELUM run penuh.

In [2]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

ROOT : /opt/shared_data/hibah-riset
DATA : /opt/shared_data/hibah-riset/data/s2
python: /home/if2011/miniconda3/envs/jupyterhub-env/bin/python


In [3]:
# sys.path untuk import diffmot
import sys
sys.path.insert(0, str(EXT / "diffmot"))
print(sys.path[:2])

['/opt/shared_data/hibah-riset/external/diffmot', '/opt/shared_data/hibah-riset/external/diffmot/external']


In [4]:
!python $S2_ROOT/scripts/s2/patch_diffmot_eval.py --diffmot-root $S2_EXT/diffmot

SUDAH TERPATCH di /opt/shared_data/hibah-riset/external/diffmot/diffmot.py — tidak ada perubahan
SUDAH TERPATCH di /opt/shared_data/hibah-riset/external/diffmot/tracker/DiffMOTtracker.py — tidak ada perubahan
NUMPY ALIAS: SUDAH BERSIH


In [5]:
# verifikasi baris hasil patch
src = (EXT / "diffmot" / "diffmot.py").read_text()
import re
for pat in ["img = cv2.imread(im_path)", "tag, img)"]:
    print(pat, "->", "OK" if pat in src else "MISSING")

img = cv2.imread(im_path) -> OK
tag, img) -> OK


### Smoke ReID model (FastReID)

In [6]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

torch 2.0.1+cu118 | cuda True
NVIDIA GeForce RTX 4090


In [7]:
# load bobot ReID DanceTrack — validasi path & arsitektur
from external.adaptors.fastreid_adaptor import FastReID
w_dance = EXT / "diffmot" / "external" / "weights" / "dance_sbs_S50.pth"
assert w_dance.exists(), f"tidak ada {w_dance} — jalankan notebook 10 dulu"
m = FastReID(str(w_dance))
m.eval(); m.cuda(); m.half()
x = torch.randn(1, 3, 384, 128).half().cuda()   # (N,C,H,W) sesuai crop_size (128,384) di embedding.py
with torch.no_grad():
    y = m(x)
print("ReID forward OK, output:", tuple(y.shape))

FileNotFoundError: [Errno 2] No such file or directory: 'external/fast_reid/configs/MOT17/sbs_S50.yml'

### Smoke pipeline embedding + cache (1 frame, 1 sekuens)

In [ ]:
import cv2, numpy as np
from tracker.embedding import EmbeddingComputer

class Cfg:
    reid_dir = str(DATA / "embeddings" / "dance")

ec = EmbeddingComputer(Cfg(), "dance", True, True)   # dataset='dance' -> pakai dance_sbs_S50.pth
seq0 = sorted(p for p in (DATA / "dancetrack" / "val").iterdir() if p.is_dir())[0]
img = cv2.imread(str(sorted((seq0 / "img1").glob("*.*"))[0]))
bbox = np.array([[10, 10, 120, 240], [200, 60, 320, 300]], np.float32)
emb = ec.compute_embedding(img, bbox, f"{seq0.name}:1")
print("embedding shape:", emb.shape)
ec.dump_cache()
print("cache:", Cfg.reid_dir)

**Lanjut**: `50_s2_run_diffmot.ipynb` (kernel `s2-diffmot`).

In [9]:
!python {ROOT}/scripts/s2/realtime_demo_diffmot.py --check

import DiffMOT: OK
python : /home/if2011/miniconda3/envs/jupyterhub-env/bin/python
torch  : 2.0.1+cu118 | cuda=True
OK — siap run.


In [14]:
!python {ROOT}/scripts/s2/realtime_demo_diffmot.py --source {ROOT}/mot20-02_clip.mp4 --save demo_mot.mp4

GPU: NVIDIA GeForce RTX 4090
Load ckpt D2MP: /opt/shared_data/hibah-riset/external/diffmot/experiments/diffmot_mot/mot_epoch800.pt
D2MP OK. Load YOLO: /opt/shared_data/hibah-riset/data/s2/weights/best.pt
Output: /opt/shared_data/hibah-riset/external/diffmot/demo_mot.mp4 | frame 1920x1080 @ 25 fps
Jalan! (headless — hasil di mp4; Ctrl+C untuk stop)
/home/febnawan2/.local/lib/python3.8/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
Sumber selesai.
Selesai. Video: /opt/shared_data/hibah-riset/external/diffmot/demo_mot.mp4 (FPS rata-rata 20.7, 450 frame)
